# Analysis of discharge

In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from config import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# Processing metadata, isolating specific parameters
md_wide = pd.read_csv(metadata_filepath+"metadata.csv", dtype = {'sourceID': str})
print(f"Full dataset size: {md_wide.shape[0]}")

for solute in solutes:
    md_wide[solute] = md_wide['WQ_parameters'].str.contains(solute)

# Removing the stations without data
md_wide = md_wide[md_wide[solutes].any(axis=1)]
print(f"Filtered dataset size: {md_wide.shape[0]}")

Full dataset size: 4268
Filtered dataset size: 4242


# Sampling Coverage Across Flow Regime

In [3]:
# Number of bins for the flow distribution (20 bins = 5% increments)
n_bins = 10
bin_edges_pct = np.linspace(0, 1, n_bins + 1)
bin_labels = [f'{int(bin_edges_pct[i]*100)}-{int(bin_edges_pct[i+1]*100)}'
              for i in range(n_bins)]
bin_edges_pct

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [ ]:
# Bulding the dataframe
results = []
missing_flow = 0
cant_bin = []

# Iterate over each row and calculate
for i, row in md_wide.iterrows():
    try:
        site = row['STREAM_ID']
        wq_para_data = pd.read_csv(water_quality_filepath+site+'.csv', low_memory=False)
        q_data  = pd.read_csv(discharge_filepath+site+'.csv', low_memory=False)

        # Removing 0 flow values ## BUG REPORTED
        q_data = q_data.loc[q_data['Q_m3s'] != 0]

        if q_data.empty:
            continue
        
    except:
        missing_flow+=1  
        continue 
    # Determine distribution of flow. This should be done with a longer period of record, but for now we will leave it. 
    Q_percentiles = np.quantile(q_data['Q_m3s'].dropna(), bin_edges_pct)
    
    # Identify which parameters are available
    wq_params = [col for col in wq_para_data.columns 
                    if col != 'DateTime' and not col.startswith('Flag_')]
    wq_params = [t for t in wq_params if t in solutes]
    
    for param in wq_params:
        wq_para_i = wq_para_data[['DateTime', param]].copy()
        wq_para_i.dropna(subset = [param], inplace=True)
        wq_para_i = wq_para_i.merge(q_data, on='DateTime', how='left')
        wq_para_i.dropna(subset=['Q_m3s'], inplace=True)
        
        if wq_para_i.empty:
            continue
        
        # Binning the sample flow based on percentile 
        try:
            wq_para_i['bins'] = pd.cut(wq_para_i['Q_m3s'],
                                        bins=Q_percentiles,
                                        labels=bin_labels,
                                        include_lowest=True,
                                        duplicates='drop')
        except: 
            cant_bin.append(site)
            continue
        
        # Count samples per flow bin
        counts = wq_para_i['bins'].value_counts().reindex(bin_labels, fill_value=0)
        
        result_row = {
            'STREAM_ID': site,
            'param': param,
            'n_samples': int(counts.sum())
        }
        for label in bin_labels:
            result_row[label] = counts[label]
        results.append(result_row)

df_results = results
print(f"Missing flow for {np.round((missing_flow/len(md_wide))*100,1)}% of stations.")

In [ ]:
print('Stations with likely errors in the flow data:')
print(np.unique(cant_bin))

In [ ]:
# Organizing the dataframe
WQ_Q_binned_results = pd.DataFrame(results)
WQ_Q_binned_fractions = WQ_Q_binned_results[bin_labels].div(WQ_Q_binned_results['n_samples'], axis=0)*100

WQ_Q_binned_fractions.insert(0, 'STREAM_ID', WQ_Q_binned_results['STREAM_ID'])
WQ_Q_binned_fractions.insert(1, 'param', WQ_Q_binned_results['param'])
WQ_Q_binned_fractions.insert(2, 'n_samples', WQ_Q_binned_results['n_samples'])

# Expected distribution
expected = 1.0 / n_bins
print(f"The expected distribution if the sampling was evenly distributed is {expected*100}%.")

WQ_Q_binned_fractions.head()

In [ ]:
# Per-solute colorbar layout + label placement
cbar_config = {
    'DO_mgL': {
        'fig_width':   4,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.2],
        'title_xy':    (0.0, 1.2),
        'subtitle_xy': (0.0, 1.04),
    },
    'SpC_uScm': {
        'fig_width':   4,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.2],
        'title_xy':    (0.0, 1.2),
        'subtitle_xy': (0.0, 1.04),
    },
    'Turb_FNU': {
        'fig_width':   4.5,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.2],
        'title_xy':    (0.0, 1.2),
        'subtitle_xy': (0.0, 1.03),
    },
    'WTemp_C': {
        'fig_width':   4,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.105],
        'title_xy':    (0.0, 1.2),
        'subtitle_xy': (0.0, 1.04),
    },
    'NO3_mgNL': {
        'fig_width':   4,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.25],
        'title_xy':    (0.0, 1.15),
        'subtitle_xy': (0.0, 1.04),
    },
    'fDOM_QSU': {
        'fig_width':   4,
        'fig_height': 10,
        'inset':       [1.025, 0.4, 0.03, 0.25],
        'title_xy':    (0.0, 1.15),
        'subtitle_xy': (0.0, 1.04),
    },
}

# Fallback so a missing key doesn't crash the loop
default_cbar_config = {
    'fig_width':   4,
    'fig_height': 15,
    'inset':       [1.025, 0.4, 0.03, 0.105],
    'title_xy':    (0.0, 1.13),
    'subtitle_xy': (0.0, 1.04),
}

In [ ]:
import seaborn as sns
from matplotlib.lines import Line2D

params = list(PARAM_COLORS.keys())
min_samples=0
long = (WQ_Q_binned_fractions[WQ_Q_binned_fractions['n_samples'] >= min_samples]
        .melt(id_vars=['STREAM_ID', 'param', 'n_samples'],
              value_vars=bin_labels, var_name='flow_bin', value_name='pct'))

fig, axes = plt.subplots(6, 1, figsize=(5, 15), sharex=True, sharey=True)

for ax, solute in zip(axes, params):
    g = long[long['param'] == solute]
    if g.empty:
        ax.set_visible(False)
        continue

    edge = PARAM_COLORS[solute]['edge']
    sns.boxplot(data=g, x='flow_bin', y='pct', order=bin_labels,
                color=PARAM_COLORS[solute]['fill'], showfliers=False, ax=ax,
                linewidth=0.8,
                boxprops=dict(edgecolor=edge),
                whiskerprops=dict(color=edge),
                capprops=dict(color=edge),
                medianprops=dict(color=edge, linewidth=1.4))

    # even-sampling reference, now clearly visible and on top of the boxes
    ax.axhline(expected * 100, ls='--', color='0.25', lw=1.1, zorder=5)

    # solute label on the left, outside the axes
    ax.annotate(f'{solute_pretty.get(solute, solute)}   (n = {g.STREAM_ID.nunique()})',
                xy=(1, 0.92), xycoords='axes fraction',
                ha='right', va='top', fontsize=9)

    ax.set_ylabel("% of samples", fontsize=8)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

# reference-line legend, top-left panel
axes[0].legend(handles=[Line2D([0], [0], ls='--', color='0.25', lw=1.1,
                               label=f'even sampling ({expected*100:.0f}%)')],
               frameon=False, fontsize=8, loc='upper left')

axes[-1].set_xlabel('Flow percentile bin')

ax = axes[-1]
ticks = range(0, len(bin_labels), 2)
ax.set_xticks(ticks)
ax.set_xticklabels([bin_labels[i] for i in ticks], rotation=0, ha='center', fontsize=7)

plt.tight_layout()
plt.subplots_adjust(hspace=0.15, left=0.15)
plt.savefig('../OUTPUT/temporal_patterns/sampling_bias_box_6panel.png',
            dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
params = list(PARAM_COLORS.keys())
min_samples = 0
long = (WQ_Q_binned_fractions[WQ_Q_binned_fractions['n_samples'] >= min_samples]
        .melt(id_vars=['STREAM_ID', 'param', 'n_samples'],
              value_vars=bin_labels, var_name='flow_bin', value_name='pct'))

fig, axes = plt.subplots(3, 2, figsize=(8, 8), sharex=True, sharey=True)
flat = axes.flatten()

for ax, solute in zip(flat, params):
    g = long[long['param'] == solute]
    if g.empty:
        ax.set_visible(False)
        continue

    edge = PARAM_COLORS[solute]['edge']
    sns.boxplot(data=g, x='flow_bin', y='pct', order=bin_labels,
                color=PARAM_COLORS[solute]['fill'], showfliers=False, ax=ax,
                linewidth=0.8,
                boxprops=dict(edgecolor=edge),
                whiskerprops=dict(color=edge),
                capprops=dict(color=edge),
                medianprops=dict(color=edge, linewidth=1.4))

    ax.axhline(expected * 100, ls='--', color='0.25', lw=1.1, zorder=5)
    ax.annotate(f'{solute_pretty.get(solute, solute)}   (n = {g.STREAM_ID.nunique()})',
                xy=(1, 0.92), xycoords='axes fraction',
                ha='right', va='top', fontsize=9)
    ax.set_ylabel("% of samples", fontsize=8)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

# reference-line legend on the first panel
flat[0].legend(handles=[Line2D([0], [0], ls='--', color='0.25', lw=1.1,
                               label=f'even sampling ({expected*100:.0f}%)')],
               frameon=False, fontsize=8, loc='upper left')

# x ticks/labels on the bottom row only (sharex hides them on upper rows)
ticks = range(0, len(bin_labels), 2)
for ax in axes[-1, :]:
    ax.set_xticks(ticks)
    ax.set_xticklabels([bin_labels[i] for i in ticks], rotation=0, ha='center', fontsize=7)
    ax.set_xlabel('Flow percentile bin')

plt.tight_layout()
plt.subplots_adjust(hspace=0.15, wspace=0.1)
plt.savefig('../OUTPUT/temporal_patterns/sampling_bias_box_32panel.png',
            dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
centers = np.array([(int(l.split('-')[0]) + int(l.split('-')[1])) / 2
                    for l in bin_labels])
exp_pct = expected * 100
color = '#3b6ea5'

params = params if 'params' in globals() else list(solute_pretty.keys())
params = params[:6]

fig, axes = plt.subplots(6, 1, figsize=(5, 15), sharex=True, sharey=True)

for ax, solute in zip(axes, params):
    g = WQ_Q_binned_fractions[(WQ_Q_binned_fractions['param'] == solute)
                              & (WQ_Q_binned_fractions['n_samples'] >= min_samples)]
    if g.empty:
        ax.set_visible(False)
        continue

    M = g[bin_labels].values                       # (n_sites, n_bins), already %
    med = np.median(M, axis=0)
    q25, q75 = np.percentile(M, [25, 75], axis=0)
    q10, q90 = np.percentile(M, [10, 90], axis=0)

    ax.fill_between(centers, q10, q90, color=color, alpha=0.15, lw=0)
    ax.fill_between(centers, q25, q75, color=color, alpha=0.30, lw=0)
    ax.plot(centers, med, color=color, lw=2.2)
    ax.axhline(exp_pct, ls='--', color='0.35', lw=1)

    ax.set_xlim(0, 100)
    ax.set_ylim([0,25])
    ax.set_ylabel("% of samples", fontsize=8)
    ax.annotate(f'{solute_pretty.get(solute, solute)}   (n = {M.shape[0]})',
                xy=(1, 0.1), xycoords='axes fraction', ha='right', va='top',
                fontsize=9)
    ax.tick_params(labelsize=7)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

axes[-1].set_xlabel('Flow percentile')

# one shared legend built from proxy handles, not per-panel labels
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], color=color, lw=2.2, label='median site'),
    Patch(facecolor=color, alpha=0.30, label='25–75th pct of sites'),
    Patch(facecolor=color, alpha=0.15, label='10–90th pct of sites'),
    Line2D([0], [0], ls='--', color='0.35', lw=1,
           label=f'even sampling ({exp_pct:.0f}%)'),
]
axes[0].legend(handles=handles, frameon=False, fontsize=7, loc='upper right')

plt.tight_layout()
plt.subplots_adjust(hspace=0.12)
plt.savefig('../OUTPUT/temporal_patterns/sampling_bias_ribbon_6panel.png',
            dpi=600, bbox_inches='tight')
plt.close(fig)